# 8장. FAISS 기반 RAG 실습

이 장은 PDF 교재의 `6.6 RAG 예제`, `6.7 RAG 프로세스`, `6.8 벡터스토어 비교` 중 FAISS 관련 내용과 `LLM/llm.ipynb`, `LLM/llm2.ipynb`의 FAISS 기반 RAG 실습을 통합한 장입니다.

## 이 장에서 다루는 내용

1. FAISS란 무엇인가
2. FAISS가 RAG에서 하는 역할
3. 텍스트 파일 기반 FAISS RAG
4. `TextLoader`
5. `HuggingFaceEmbeddings`
6. `FAISS.from_documents`
7. Retriever 설정
8. Ollama Gemma2 모델 연결
9. LCEL 방식 RAG Chain 구성
10. 질문 실행
11. 짧은 문장 기반 FAISS 검색 예제
12. CSV 기반 FAISS RAG 챗봇
13. 출처 표시
14. FAISS 사용 시 오류와 개선 포인트

7장에서 RAG의 전체 흐름을 배웠다면, 8장에서는 FAISS를 사용해 실제 검색증강생성 체인을 구현합니다.


## 8.1 FAISS란?

FAISS는 Facebook AI Similarity Search의 약자로, 대량의 벡터에서 유사한 벡터를 빠르게 찾기 위한 고성능 벡터 검색 라이브러리입니다.

PDF 6.8의 벡터스토어 비교에서 FAISS는 다음처럼 설명됩니다.

| 항목 | 설명 |
|---|---|
| 주요 특징 | 고성능 벡터 검색 라이브러리입니다. 독립형 DB는 아니며 보조 저장소가 필요할 수 있습니다. |
| 특장점 | 매우 빠른 검색 속도와 대규모 데이터셋 처리에 최적화되어 있습니다. |
| 주요 기능 | 코사인, 유클리드 등 다양한 거리 함수를 지원합니다. |

FAISS는 개발자가 직접 벡터 인덱스를 만들고 검색하는 데 강하지만, Chroma처럼 문서 관리와 영속 저장을 포함한 DB형 사용성은 상대적으로 약합니다.


## 8.2 FAISS 기반 RAG 흐름

FAISS RAG의 기본 흐름은 다음과 같습니다.

```text
1. 문서 로드
2. 문서 임베딩
3. FAISS 벡터스토어 생성
4. Retriever 변환
5. 사용자 질문 입력
6. 관련 문서 검색
7. Prompt 구성
8. Ollama LLM 답변 생성
```

`LLM/llm.ipynb`의 FAISS RAG 예제는 PDF 6.6의 예제와 거의 같은 구조입니다.


## 8.3 설치 준비

원본 노트북의 FAISS RAG 예제 설치 주석은 다음과 같습니다.

```text
pip install langchain langchain-community langchain-text-splitters sentence-transformers faiss-cpu
```

필요 패키지 역할:

| 패키지 | 역할 |
|---|---|
| `langchain` | LangChain 기본 프레임워크 |
| `langchain-community` | TextLoader, FAISS, Ollama 등 community 통합 모듈 |
| `langchain-core` | Prompt, Runnable, OutputParser 등 핵심 구성요소 |
| `sentence-transformers` | Hugging Face 임베딩 모델 실행 |
| `faiss-cpu` | FAISS 벡터 검색 라이브러리 |
| `langchain-text-splitters` | 문서 분할기 |


In [ ]:
# FAISS RAG 실습용 설치 예시입니다.
# 필요한 경우 주석을 해제하고 실행하세요.

# %pip install langchain langchain-community langchain-core langchain-text-splitters
# %pip install sentence-transformers faiss-cpu


## 8.4 텍스트 파일 기반 FAISS RAG: 모듈 import

`llm.ipynb`의 FAISS RAG 예제는 다음 모듈을 사용합니다.

| 모듈 | 역할 |
|---|---|
| `ChatPromptTemplate` | RAG 프롬프트 템플릿 구성 |
| `StrOutputParser` | LLM 응답을 문자열로 파싱 |
| `RunnablePassthrough` | 질문을 그대로 프롬프트에 전달 |
| `TextLoader` | 텍스트 파일 로드 |
| `FAISS` | 벡터스토어 생성 |
| `HuggingFaceEmbeddings` | 문서와 질문 임베딩 |
| `Ollama` | 로컬 LLM 답변 생성 |
| `RecursiveCharacterTextSplitter` | 문서 분할, 이 예제에서는 import만 되어 있음 |


In [ ]:
# 운영체제 경로 처리 등에 사용할 수 있습니다.
import os

# 프롬프트 템플릿을 만들기 위한 LangChain 클래스입니다.
from langchain_core.prompts import ChatPromptTemplate

# LLM 응답을 문자열로 정리하는 출력 파서입니다.
from langchain_core.output_parsers import StrOutputParser

# 입력 질문을 그대로 다음 단계로 전달합니다.
from langchain_core.runnables import RunnablePassthrough

# 텍스트 파일을 LangChain Document로 로드합니다.
from langchain_community.document_loaders import TextLoader

# FAISS 벡터스토어입니다.
from langchain_community.vectorstores import FAISS

# Hugging Face 기반 임베딩 모델입니다.
from langchain_community.embeddings import HuggingFaceEmbeddings

# 로컬 Ollama LLM을 LangChain에서 사용합니다.
from langchain_community.llms import Ollama

# 문서를 청크로 나눌 때 사용하는 분할기입니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter


## 8.5 1단계: 데이터 로드

PDF 6.7의 첫 단계는 문서 로딩입니다.

`llm.ipynb`에서는 다음 파일을 읽습니다.

```text
./dataset/history.txt
```

원본 코드:

```python
loader = TextLoader("./dataset/history.txt", encoding='UTF8')
documents = loader.load()
```

`TextLoader`는 텍스트 파일을 LangChain Document 객체 목록으로 읽어옵니다. 각 Document는 `page_content`와 `metadata`를 가집니다.


In [ ]:
# 1. 데이터 로드
# history.txt 파일을 UTF-8 인코딩으로 읽습니다.
loader = TextLoader("./dataset/history.txt", encoding='UTF8')

# 텍스트 파일을 Document 객체 목록으로 로드합니다.
documents = loader.load()

# 로드된 문서 개수를 확인합니다.
print("로드된 문서 수:", len(documents))

# 첫 문서의 일부 내용을 확인합니다.
print(documents[0].page_content[:300])


## 8.6 2~4단계: 임베딩 생성과 FAISS 저장

PDF 6.7에서 문서 로딩 후에는 문서 분할, 임베딩 변환, 벡터스토어 저장을 진행합니다.

`llm.ipynb`의 간단 FAISS 예제는 문서 분할 없이 로드된 문서를 바로 FAISS에 넣습니다.

```python
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents, embeddings)
```

이때 내부적으로 각 문서가 임베딩 벡터로 변환되고, FAISS 인덱스에 저장됩니다.

주의: 문서가 길면 실제 프로젝트에서는 먼저 청크로 분할하는 것이 좋습니다.


In [ ]:
# 2. 벡터 임베딩 생성
# all-MiniLM-L6-v2 모델은 문장을 벡터로 바꾸는 sentence-transformers 모델입니다.
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 문서들을 임베딩한 뒤 FAISS 벡터스토어로 저장합니다.
vectorstore = FAISS.from_documents(documents, embeddings)

# 필요하다면 로컬에 FAISS 인덱스를 저장할 수 있습니다.
# vectorstore.save_local("faiss_index")


## 8.7 5~6단계: Retriever 설정

FAISS 벡터스토어를 만들었다면, 질문에 대해 관련 문서를 검색할 수 있도록 Retriever로 변환합니다.

원본 코드:

```python
retriever = vectorstore.as_retriever()
```

Retriever는 다음 역할을 합니다.

1. 사용자 질문을 임베딩합니다.
2. FAISS 인덱스에서 유사한 문서 벡터를 찾습니다.
3. 관련 Document 객체를 반환합니다.


In [ ]:
# 3. 검색기 설정
# FAISS 벡터스토어를 Retriever로 변환합니다.
retriever = vectorstore.as_retriever()


In [ ]:
# Retriever가 어떤 문서를 찾는지 미리 확인할 수 있습니다.
sample_docs = retriever.invoke("고조선은 언제 설립되었는지 알려줘.")

# 검색된 문서 일부를 출력합니다.
for i, doc in enumerate(sample_docs, start=1):
    print(f"--- 검색 문서 {i} ---")
    print(doc.page_content[:300])


## 8.8 7~8단계: Ollama LLM 초기화

검색된 문서를 기반으로 답변을 생성할 LLM을 준비합니다.

원본 노트북은 Ollama의 Gemma2 모델을 사용합니다.

```python
llm = Ollama(model="gemma2", base_url="http://localhost:11434")
```

`base_url`은 Ollama 로컬 서버 주소입니다.

Ollama가 실행 중이어야 하며, `gemma2` 모델이 설치되어 있어야 합니다.


In [ ]:
# 4. Ollama Gemma2 모델 초기화
# Ollama 서버가 localhost:11434에서 실행 중이어야 합니다.
llm = Ollama(model="gemma2", base_url="http://localhost:11434")


## 8.9 RAG 프롬프트 구성

RAG에서 프롬프트는 매우 중요합니다.

원본 노트북의 프롬프트는 다음 규칙을 포함합니다.

- 당신은 질문에 답변하는 AI 어시스턴트입니다.
- 제공된 context만을 바탕으로 질문에 답변하세요.
- 모르면 모른다고 답하세요.

이 지시문은 검색된 문서 밖의 내용을 모델이 마음대로 생성하는 것을 줄이는 데 도움을 줍니다.


In [ ]:
# 5.1 프롬프트 템플릿 정의
template = """
당신은 질문에 답변하는 AI 어시스턴트입니다.
제공된 context만을 바탕으로 질문에 답변하세요. 모르면 모른다고 답하세요.

[Context]
{context}

[Question]
{question}
"""

# 문자열 템플릿을 LangChain ChatPromptTemplate로 변환합니다.
prompt = ChatPromptTemplate.from_template(template)


## 8.10 LCEL 방식 RAG Chain 구성

원본 노트북은 `RetrievalQA` 대신 LCEL 방식으로 RAG 체인을 구성합니다.

```python
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
```

각 단계의 의미:

| 단계 | 역할 |
|---|---|
| `context: retriever` | 질문으로 관련 문서를 검색합니다. |
| `question: RunnablePassthrough()` | 질문 원문을 그대로 전달합니다. |
| `prompt` | 검색 context와 질문을 하나의 프롬프트로 만듭니다. |
| `llm` | Ollama Gemma2가 답변을 생성합니다. |
| `StrOutputParser()` | 결과를 문자열로 정리합니다. |


In [ ]:
# 5.2 LCEL 체인 조합
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


## 8.11 질문 실행

PDF 6.6과 원본 노트북의 질문 예시는 다음과 같습니다.

```text
고조선은 언제 설립되었는지 알려줘.
```

예상 답변은 history 문서 내용에 따라 달라지지만, PDF 예제의 결과값은 다음과 같습니다.

```text
고조선은 기원전 2333년 단군왕검에 의해 세워졌다고 전해집니다.
```


In [ ]:
# 6. 질문 실행
query = "고조선은 언제 설립되었는지 알려줘."

try:
    # RAG 체인을 실행합니다.
    response = rag_chain.invoke(query)

    # 질문과 답변을 출력합니다.
    print("[질문]:", query)
    print("[답변]:", response)
except Exception as e:
    # 실행 중 오류가 발생하면 메시지를 출력합니다.
    print(f"RAG 체인 실행 중 오류: {e}")


## 8.12 짧은 문장 기반 FAISS 검색 예제

`llm.ipynb`에는 파일을 읽지 않고 짧은 문장 리스트를 바로 FAISS에 넣는 예제도 있습니다.

이 예제는 RAG 구조를 작게 이해하기 좋습니다.

문서:

- 영준은 랭체인 주식회사에서 근무를 하였습니다.
- 설현은 테디와 같은 회사에서 근무하였습니다.
- 영준의 직업은 개발자입니다.
- 설현의 직업은 디자이너입니다.

질문:

- 설현의 직업은?
- 영준이 근무한 곳은?
- 영준이 개발자라면 유망한 SW 기술 추천해줘.

마지막 질문은 문서에 직접 답이 없는 질문이라, 검색 문맥과 LLM 자체 지식이 섞일 수 있습니다. 이는 RAG에서 프롬프트 통제와 검색 품질이 중요하다는 점을 보여줍니다.


In [ ]:
# 짧은 문장 리스트를 FAISS 벡터스토어로 만듭니다.
mini_vectorstore = FAISS.from_texts(
    [
        "영준은 랭체인 주식회사에서 근무를 하였습니다.",
        "설현은 테디와 같은 회사에서 근무하였습니다.",
        "영준의 직업은 개발자입니다.",
        "설현의 직업은 디자이너입니다.",
    ],
    embedding=HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask'),
)

# Retriever로 변환합니다.
mini_retriever = mini_vectorstore.as_retriever()


In [ ]:
# 짧은 문장 검색용 프롬프트입니다.
mini_template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

# 프롬프트 객체를 만듭니다.
mini_prompt = ChatPromptTemplate.from_template(mini_template)

# Ollama 모델을 연결합니다.
mini_model = Ollama(model="gemma2")

# 검색 체인을 구성합니다.
mini_retrieval_chain = (
    {"context": mini_retriever, "question": RunnablePassthrough()}
    | mini_prompt
    | mini_model
    | StrOutputParser()
)


In [ ]:
# 설현의 직업을 검색 문맥에서 찾아 답합니다.
mini_retrieval_chain.invoke("설현의 직업은?")


In [ ]:
# 영준이 근무한 곳을 검색 문맥에서 찾아 답합니다.
mini_retrieval_chain.invoke("영준이 근무한 곳은?")


In [ ]:
# 문서에 없는 확장 질문입니다.
mini_retrieval_chain.invoke("영준이 개발자라면 유망한 SW 기술 추천해줘. 한글로 설명해줘")


## 8.13 CSV 기반 FAISS RAG 챗봇

`llm2.ipynb`에는 CSV 파일을 읽어 FAISS 벡터스토어로 만들고, Gradio 챗봇으로 제공하는 예제가 있습니다.

사용 파일:

```text
./dataset/indata_kor.csv
```

인코딩:

```python
encoding='CP949'
```

이 예제는 PDF 6.4의 문서 기반 질의응답, 헬프데스크/챗봇 활용 사례와 연결됩니다.


In [ ]:
# CSV RAG에 필요한 모듈입니다.
import pandas as pd
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.prompts import MessagesPlaceholder
import gradio as gr


In [ ]:
# CSV 파일을 로드합니다.
df = pd.read_csv("./dataset/indata_kor.csv", encoding='CP949')

# 데이터 일부를 확인합니다.
df.tail()


In [ ]:
# DataFrame 내용을 문자열로 바꾼 뒤 청크로 분할합니다.
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_text("\n".join(df.to_string()))

# 임베딩 모델을 초기화합니다.
csv_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# FAISS 벡터스토어를 생성합니다.
csv_vectorstore = FAISS.from_texts(texts, csv_embeddings)

# 검색기는 가장 관련 있는 청크 1개를 반환하도록 설정합니다.
csv_retriever = csv_vectorstore.as_retriever(search_kwargs={"k": 1})


In [ ]:
# CSV RAG에 사용할 ChatOllama 모델입니다.
csv_llm = ChatOllama(model="gemma2", temperature=0.1)

# 대화 이력과 context를 함께 쓰는 프롬프트입니다.
csv_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer based on the provided context."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}\n\nContext: {context}")
])


In [ ]:
# 검색된 문서들을 문자열로 변환합니다.
def format_docs(docs):
    if not docs:
        return "No context available"

    result = []
    for doc in docs:
        if hasattr(doc, 'page_content'):
            result.append(doc.page_content)

    return "\n".join(result)


In [ ]:
# CSV RAG 챗봇 함수입니다.
def csv_chat(message, history):
    # Gradio의 history를 LangChain 메시지로 변환합니다.
    chat_history = []
    for human, ai in history:
        chat_history.append(HumanMessage(content=human))
        chat_history.append(AIMessage(content=ai))

    # 질문과 관련된 문서를 검색합니다.
    docs = csv_retriever.invoke(message)
    context = format_docs(docs)

    # 프롬프트 메시지를 구성합니다.
    messages = csv_prompt.format_messages(
        chat_history=chat_history,
        question=message,
        context=context
    )

    # LLM으로 답변을 생성합니다.
    response = csv_llm.invoke(messages)

    # 출처 정보를 metadata에서 추출합니다.
    sources = set([doc.metadata.get('source', 'Unknown') for doc in docs])
    source_info = f"\n\n참고 출처: {', '.join(sources)}" if sources else ""

    # 답변과 출처 정보를 반환합니다.
    return response.content + source_info


In [ ]:
# CSV 기반 대학 정보 챗봇 UI입니다.
csv_demo = gr.ChatInterface(
    fn=csv_chat,
    examples=[
        "한국폴리텍대학 스마트금융과 입학 전까지 어떤걸 공부하면 될까요?",
        "스마트금융과에 대해 설명해주세요",
        "한국폴리텍대한 추천할만한 학과 하나를 소개해주세요."
    ],
    title="대학 정보 AI 챗봇",
    description="스마트금융과에 대한 질문을 입력하면 AI가 CSV 데이터를 참고하여 한글로 답변합니다."
)


In [ ]:
# CSV RAG 챗봇 서버를 실행합니다.
csv_demo.launch(server_port=7861, server_name="0.0.0.0")


In [ ]:
# CSV RAG 챗봇 서버를 종료합니다.
csv_demo.close()


## 8.14 FAISS RAG 개선 포인트

기본 FAISS RAG를 더 좋게 만들려면 다음을 조정할 수 있습니다.

### 1. 문서 분할 추가

긴 문서는 `RecursiveCharacterTextSplitter`로 청크 분할 후 FAISS에 넣는 것이 좋습니다.

### 2. 검색 개수 조정

`search_kwargs={"k": 3}`처럼 관련 문서 개수를 조정할 수 있습니다.

### 3. 한국어 임베딩 모델 선택

한국어 문서에는 한국어 또는 다국어 임베딩 모델이 더 적합할 수 있습니다.

### 4. 프롬프트 강화

`모르면 모른다고 답하세요`, `context에 없는 내용은 추측하지 마세요` 같은 지시를 추가합니다.

### 5. 출처 표시

Document metadata를 활용해 어떤 문서에서 근거를 가져왔는지 표시합니다.

### 6. 인덱스 저장

FAISS 인덱스를 매번 새로 만들지 않고 `save_local()`과 `load_local()`을 활용할 수 있습니다.


## 8.15 자주 발생하는 오류와 해결 방법

### 1. `faiss` 설치 오류

Windows에서는 보통 다음 패키지를 설치합니다.

```python
%pip install faiss-cpu
```

### 2. 임베딩 모델 다운로드 지연

처음 `HuggingFaceEmbeddings`를 실행하면 모델 다운로드가 필요합니다. 네트워크 상태에 따라 시간이 걸릴 수 있습니다.

### 3. `history.txt` 파일 없음

상대 경로 기준입니다. 노트북 실행 위치가 `C:/work/LLM`인지 확인합니다.

### 4. CSV 인코딩 오류

`indata_kor.csv`는 원본 노트북에서 `CP949`로 읽습니다. 오류가 나면 `utf-8`, `utf-8-sig`도 확인합니다.

### 5. Ollama 연결 실패

Ollama 앱이 실행 중인지, `gemma2` 모델이 설치되어 있는지 확인합니다.

### 6. 답변이 문서와 무관함

가능한 원인:

- 검색된 문서가 질문과 관련 없음
- 임베딩 모델이 문서 언어에 적합하지 않음
- 청크가 너무 크거나 작음
- 프롬프트가 context 기반 답변을 충분히 강제하지 않음


## 8.16 이 장의 핵심 정리

이 장에서 배운 내용을 정리하면 다음과 같습니다.

- FAISS는 빠른 벡터 검색 라이브러리입니다.
- FAISS는 RAG에서 문서 임베딩 벡터를 저장하고 유사 문서를 찾는 역할을 합니다.
- `TextLoader`로 텍스트 파일을 읽을 수 있습니다.
- `HuggingFaceEmbeddings`로 문서를 벡터로 변환합니다.
- `FAISS.from_documents()` 또는 `FAISS.from_texts()`로 벡터스토어를 만듭니다.
- `as_retriever()`로 질문 기반 검색기를 만듭니다.
- `RunnablePassthrough`, `ChatPromptTemplate`, `Ollama`, `StrOutputParser`를 연결해 LCEL RAG Chain을 구성합니다.
- CSV 데이터도 문자열로 변환하고 청크 분할한 뒤 FAISS RAG에 사용할 수 있습니다.
- FAISS RAG의 품질은 임베딩 모델, 청크 전략, 검색 개수, 프롬프트 설계에 크게 좌우됩니다.

다음 장에서는 Chroma 기반 RAG를 구현하고, FAISS와 Chroma의 차이를 실습 중심으로 살펴봅니다.
